[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zhangdongming0607/ai-memory-course/blob/main/02-talk-to-llm.ipynb)

> 点击上方按钮，在 Google Colab 中打开本课，可以直接运行代码，无需本地环境。

# 第 2 课：和 LLM 对话 — API 调用实战

> 学完这节课，你会：用 Python 调 OpenAI API 发消息、理解多轮对话怎么工作、知道 AI 为什么"没有记忆"。

---

## 2.1 环境准备

如果你还没装依赖，先运行：

In [ ]:
!pip install openai python-dotenv -q

In [ ]:
import os
from dotenv import load_dotenv

# 从 .env 文件加载 API Key
load_dotenv()

# 检查是否设置了 API Key
api_key = os.getenv("OPENAI_API_KEY")
if api_key:
    print(f"API Key 已加载（前8位: {api_key[:8]}...）")
else:
    print("⚠️ 未找到 OPENAI_API_KEY")
    print("请在课程目录创建 .env 文件，内容为：")
    print("OPENAI_API_KEY=sk-xxx")

## 2.2 第一次调用：发一条消息

调 LLM API 其实就像调普通 REST API。OpenAI 的 SDK 封装了 HTTP 请求，你只需要传消息列表。

In [ ]:
from openai import OpenAI

client = OpenAI()  # 自动从环境变量读取 OPENAI_API_KEY

# 最基本的调用
response = client.chat.completions.create(
    model="gpt-4o-mini",  # 便宜的模型，练习用
    messages=[
        {"role": "user", "content": "用一句话解释什么是记忆系统"}
    ]
)

# 取出 AI 的回复
print(response.choices[0].message.content)

In [ ]:
# 看看完整的响应结构
print("模型:", response.model)
print("Token 用量:", response.usage)  # 输入 token + 输出 token
print("\n完整响应:")
print(response)

### 对比前端的 fetch 调用

上面的代码，等价于这样的 HTTP 请求（前端视角）：

```javascript
const response = await fetch('https://api.openai.com/v1/chat/completions', {
  method: 'POST',
  headers: {
    'Content-Type': 'application/json',
    'Authorization': 'Bearer sk-xxx'
  },
  body: JSON.stringify({
    model: 'gpt-4o-mini',
    messages: [
      { role: 'user', content: '用一句话解释什么是记忆系统' }
    ]
  })
});
const data = await response.json();
console.log(data.choices[0].message.content);
```

本质上就是一个 POST 请求。Python SDK 只是帮你封装了。

## 2.3 加上 System Prompt

上一课说过，System Prompt 设定 AI 的"身份"。看看加不加的区别：

In [ ]:
# 不加 system prompt
response1 = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "user", "content": "帮我回复：'周末去爬山吗？'"}
    ]
)
print("【无 system prompt】")
print(response1.choices[0].message.content)
print()

In [ ]:
# 加上 system prompt + 联系人信息
response2 = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {
            "role": "system", 
            "content": """你是一个帮用户生成聊天回复的助手。

关于这个联系人的信息：
- 姓名：小明
- 关系：大学同学，关系很好
- 他喜欢户外运动，尤其是爬山和跑步
- 你们经常一起运动
- 上次一起爬了香山

请生成 3 条自然的回复建议，语气轻松随意，像朋友之间聊天。"""
        },
        {"role": "user", "content": "帮我回复：'周末去爬山吗？'"}
    ]
)
print("【有 system prompt + 联系人信息】")
print(response2.choices[0].message.content)

### 看到区别了吗？

加了联系人信息后，AI 的回复会提到"上次香山"、"跑步"等细节。

**这就是你们记忆系统的核心价值**：让 AI "知道" 对面是谁，回复更有针对性。

System Prompt 里那段"关于这个联系人的信息"就是**记忆注入（Memory Injection）**。

## 2.4 多轮对话：AI 为什么"没有记忆"

这是一个非常重要的概念：**LLM 本身不记任何东西**。

每次 API 调用都是独立的。AI 不知道你之前说过什么。

In [ ]:
# 第一轮对话
r1 = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "user", "content": "我叫小李，记住我的名字"}
    ]
)
print("第一轮:", r1.choices[0].message.content)
print()

In [ ]:
# 第二轮对话（新的 API 调用）
r2 = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "user", "content": "我叫什么名字？"}
    ]
)
print("第二轮:", r2.choices[0].message.content)
print("\n→ AI 忘了！因为第二次调用是全新的，它不知道之前说过什么")

### 那 ChatGPT 怎么"记住"上下文的？

答案很简单：**每次都把历史消息全部重新发一遍**。

In [ ]:
# 正确的多轮对话：把历史消息带上
r3 = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "user", "content": "我叫小李，记住我的名字"},
        {"role": "assistant", "content": "好的，我记住了，你叫小李！"},  # 上一轮 AI 的回复
        {"role": "user", "content": "我叫什么名字？"},  # 新问题
    ]
)
print("带历史的第二轮:", r3.choices[0].message.content)
print("\n→ 这次记住了！因为我们把第一轮对话也发给了 API")

### 这就是为什么需要记忆系统

```
方案 A（最笨）：每次都把所有历史聊天记录发过去
  → 问题：context window 装不下，费用爆炸

方案 B（最聪明）：提取关键信息存起来，每次只发摘要
  → 这就是"记忆系统"在做的事
```

对比：
```
方案 A: 发送 3000 条原始消息 ≈ 120K tokens ≈ $0.30/次
方案 B: 发送一段 500 字的摘要 ≈ 700 tokens ≈ $0.002/次
```

**150 倍的成本差距。** 这就是记忆系统的经济学。

## 2.5 Temperature 的效果对比

In [ ]:
prompt = "用一句话形容秋天"

for temp in [0, 0.5, 1.0]:
    print(f"\n--- temperature = {temp} ---")
    # 用同样的 prompt 调 3 次，看看结果是否一样
    for i in range(3):
        r = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": prompt}],
            temperature=temp,
        )
        print(f"  第{i+1}次: {r.choices[0].message.content}")

你会发现：
- `temperature=0`：三次结果几乎一样
- `temperature=0.5`：有些变化但主题一致
- `temperature=1.0`：每次都不太一样

## 2.6 动手练习

试着修改下面的代码，体验不同参数的效果：

In [ ]:
# 练习：修改 system_prompt 和 user_message，看看效果

system_prompt = "你是一个帮用户回复微信消息的助手。语气要自然随意，像朋友聊天。"

user_message = "帮我回复：'明天一起吃饭吧'"

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_message}
    ],
    temperature=0.7,  # 试试改成 0 或 1
)

print(response.choices[0].message.content)
print(f"\n消耗 token: 输入 {response.usage.prompt_tokens} + 输出 {response.usage.completion_tokens} = 总计 {response.usage.total_tokens}")

### 练习建议

1. 把 `system_prompt` 改成中文，加上联系人的信息，看看回复质量的变化
2. 试试在 `system_prompt` 里加上 "请生成 3 条建议"，看 AI 会不会遵守
3. 改变 `temperature`，对比 0 和 1 的回复风格差异

## 2.7 本课小结

| 概念 | 你学到了什么 |
|------|----------|
| API 调用 | LLM API 就是一个 POST 请求，传消息列表，返回文本 |
| System Prompt | 加了联系人信息的 system prompt 能让回复更有针对性 |
| 多轮对话 | AI 本身没有记忆，每次要把历史消息重发 |
| 记忆的价值 | 把海量聊天记录压缩成摘要，省 token 省钱 |
| Temperature | 低温度=确定性高，高温度=创意高 |

### 关键认识

**LLM 是无状态的。** 就像 HTTP 协议一样——每个请求都是独立的。
所有"记忆"都需要你自己管理，然后塞到下一次请求里。

### 下节预告

下一课我们学怎么让 AI 输出结构化的 JSON，从聊天记录中提取人物信息——这就是你们系统 OCR 识别在做的事。